In [3]:
import os

# Download real EMBER 2018 vectorized dataset
print("Downloading EMBER 2018 dataset...")
os.system('wget -q https://ember.elastic.co/ember_dataset_2018_2.tar.bz2 -O /kaggle/working/ember.tar.bz2')

print("Extracting...")
os.system('tar -xjf /kaggle/working/ember.tar.bz2 -C /kaggle/working/')

print("Listing extracted files...")
for dirname, _, filenames in os.walk('/kaggle/working'):
    for filename in filenames:
        if not filename.endswith('.tar.bz2'):
            print(os.path.join(dirname, filename))

Extracting...
Listing extracted files...
/kaggle/working/ember2018/train_features_1.jsonl
/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [7]:
import os
import sys

# Install ember library
os.system('pip install ember -q')

import ember
import numpy as np

print("Loading EMBER 2018 features...")
print("This converts raw JSONL to feature vectors...")
print("Takes 2-3 minutes...\n")

# Vectorize the dataset
ember.create_vectorized_features(
    '/kaggle/working/ember2018/'
)

# Load into numpy arrays
X_train, y_train, X_test, y_test = ember.read_vectorized_features(
    '/kaggle/working/ember2018/'
)

# Remove unlabeled samples (label = -1)
train_mask = y_train != -1
X_train = X_train[train_mask]
y_train = y_train[train_mask]

test_mask = y_test != -1
X_test = X_test[test_mask]
y_test = y_test[test_mask]

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Testing samples  : {X_test.shape[0]:,}")
print(f"Feature size     : {X_train.shape[1]}")
print(f"Malware in train : {(y_train==1).sum():,}")
print(f"Benign in train  : {(y_train==0).sum():,}")
print("\nReal EMBER dataset loaded successfully!")

ERROR: Could not find a version that satisfies the requirement ember (from versions: none)
ERROR: No matching distribution found for ember


ModuleNotFoundError: No module named 'ember'

In [8]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

print("Loading EMBER 2018 directly from JSONL files...")
print("No library needed — reading raw features\n")

def load_ember_jsonl(filepath, max_samples=None):
    features = []
    labels   = []
    
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            
            record = json.loads(line)
            label  = record.get('label', -1)
            
            # Skip unlabeled samples
            if label == -1:
                continue
            
            # Extract numeric features
            feature_vec = []
            
            # Byte histogram (256 features)
            if 'histogram' in record:
                feature_vec.extend(record['histogram'])
            
            # Byte entropy (256 features)
            if 'byteentropy' in record:
                feature_vec.extend(record['byteentropy'])
            
            # String features (104 features)
            if 'strings' in record:
                s = record['strings']
                feature_vec.extend([
                    s.get('numstrings', 0),
                    s.get('avlength', 0),
                    s.get('printabledist', [0]*96).__len__(),
                    s.get('entropy', 0),
                ])
                if 'printabledist' in s:
                    feature_vec.extend(s['printabledist'])
            
            # General file info (10 features)
            if 'general' in record:
                g = record['general']
                feature_vec.extend([
                    g.get('size', 0),
                    g.get('vsize', 0),
                    g.get('has_debug', 0),
                    g.get('exports', 0),
                    g.get('imports', 0),
                    g.get('has_relocations', 0),
                    g.get('has_resources', 0),
                    g.get('has_signature', 0),
                    g.get('has_tls', 0),
                    g.get('symbols', 0),
                ])
            
            if len(feature_vec) > 0:
                features.append(feature_vec)
                labels.append(label)
    
    return features, labels


# Load training files
print("Loading training files...")
all_features = []
all_labels   = []

# Load all 6 training files
for i in range(6):
    filepath = f'/kaggle/working/ember2018/train_features_{i}.jsonl'
    print(f"Loading train_features_{i}.jsonl...")
    feats, labs = load_ember_jsonl(filepath, max_samples=20000)
    all_features.extend(feats)
    all_labels.extend(labs)
    print(f"  Loaded {len(feats)} samples")

print(f"\nTotal samples loaded: {len(all_features)}")

# Pad features to same length
max_len = max(len(f) for f in all_features)
print(f"Padding features to length: {max_len}")

all_features = [
    f + [0] * (max_len - len(f)) 
    for f in all_features
]

X = np.array(all_features, dtype=np.float32)
y = np.array(all_labels,   dtype=np.float32)

print(f"\nFinal dataset shape : {X.shape}")
print(f"Feature dimensions  : {X.shape[1]}")
print(f"Malware samples     : {(y==1).sum():,}")
print(f"Benign samples      : {(y==0).sum():,}")
print("\nEMBER dataset loaded successfully!")

# Normalize features to [0,1]
from sklearn.preprocessing import MinMaxScaler
scaler  = MinMaxScaler()
X       = scaler.fit_transform(X)
print("Features normalized to [0,1]")

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print("\nReady to train!")

Loading EMBER 2018 directly from JSONL files...
No library needed — reading raw features

Loading training files...
Loading train_features_0.jsonl...
  Loaded 20000 samples
Loading train_features_1.jsonl...
  Loaded 14617 samples
Loading train_features_2.jsonl...
  Loaded 14605 samples
Loading train_features_3.jsonl...
  Loaded 14569 samples
Loading train_features_4.jsonl...
  Loaded 14797 samples
Loading train_features_5.jsonl...
  Loaded 14760 samples

Total samples loaded: 93348
Padding features to length: 622

Final dataset shape : (93348, 622)
Feature dimensions  : 622
Malware samples     : 38,156
Benign samples      : 55,192

EMBER dataset loaded successfully!
Features normalized to [0,1]

Training samples : 74,678
Testing samples  : 18,670

Ready to train!


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

print("Fixing feature dimensions...")

# Find the most common length
lengths = [len(f) for f in all_features]
target_len = 634  # fix to exactly 634

print(f"Target feature size: {target_len}")

# Force all features to exactly 634
fixed_features = []
for f in all_features:
    if len(f) >= target_len:
        fixed_features.append(f[:target_len])  # trim if too long
    else:
        fixed_features.append(f + [0]*(target_len - len(f)))  # pad if too short

print("Converting to numpy array...")
X = np.array(fixed_features, dtype=np.float32)
y = np.array(all_labels,     dtype=np.float32)

print(f"\nFinal dataset shape : {X.shape}")
print(f"Feature dimensions  : {X.shape[1]}")
print(f"Malware samples     : {(y==1).sum():,}")
print(f"Benign samples      : {(y==0).sum():,}")

# Normalize
print("\nNormalizing features...")
scaler = MinMaxScaler()
X      = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

FEATURE_SIZE = X_train.shape[1]

# Convert to PyTorch tensors and DataLoaders
import torch
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train),
        torch.tensor(y_train)
    ),
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test),
        torch.tensor(y_test)
    ),
    batch_size=256,
    shuffle=False
)

print(f"\nTraining samples    : {len(X_train):,}")
print(f"Testing samples     : {len(X_test):,}")
print(f"Feature size        : {FEATURE_SIZE}")
print(f"Training batches    : {len(train_loader):,}")
print("\n600,000 real EMBER samples ready!")
print("Let's build the GAN!")

Fixing feature dimensions...
Target feature size: 634
Converting to numpy array...


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (600000, 634) + inhomogeneous part.

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset

print("Flattening and fixing all features...")

TARGET = 634
fixed  = []

for f in all_features:
    # Flatten any nested lists
    flat = []
    for item in f:
        if isinstance(item, (list, tuple)):
            flat.extend(item)
        else:
            flat.append(float(item))
    
    # Trim or pad to TARGET
    if len(flat) >= TARGET:
        flat = flat[:TARGET]
    else:
        flat = flat + [0.0] * (TARGET - len(flat))
    
    fixed.append(flat)

print("Converting to numpy...")
X = np.array(fixed,      dtype=np.float32)
y = np.array(all_labels, dtype=np.float32)

print(f"Shape : {X.shape}")
print(f"Malware : {(y==1).sum():,}")
print(f"Benign  : {(y==0).sum():,}")

# Normalize
scaler = MinMaxScaler()
X      = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

FEATURE_SIZE = X_train.shape[1]

# DataLoaders
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train),
        torch.tensor(y_train)
    ),
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test),
        torch.tensor(y_test)
    ),
    batch_size=256,
    shuffle=False
)

print(f"\nTraining samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print(f"Feature size     : {FEATURE_SIZE}")
print(f"Batches          : {len(train_loader):,}")
print("\n✅ 600,000 real EMBER samples ready!")

Flattening and fixing all features...


ValueError: could not convert string to float: 'WINDOWS_GUI'

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset
import json

print("Reloading EMBER with numeric-only features...")

def load_ember_numeric(filepath, max_samples=None):
    features = []
    labels   = []
    
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            
            record = json.loads(line)
            label  = record.get('label', -1)
            
            if label == -1:
                continue
            
            flat = []
            
            # Byte histogram (256 numeric values)
            if 'histogram' in record:
                h = record['histogram']
                if isinstance(h, list):
                    flat.extend([float(x) for x in h[:256]])
            
            # Byte entropy (256 numeric values)
            if 'byteentropy' in record:
                b = record['byteentropy']
                if isinstance(b, list):
                    flat.extend([float(x) for x in b[:256]])
            
            # String features (numeric only)
            if 'strings' in record:
                s = record['strings']
                flat.append(float(s.get('numstrings', 0)))
                flat.append(float(s.get('avlength',   0)))
                flat.append(float(s.get('entropy',    0)))
                flat.append(float(s.get('paths',      0)))
                flat.append(float(s.get('urls',       0)))
                flat.append(float(s.get('registry',   0)))
                flat.append(float(s.get('MZ',         0)))
                pd = s.get('printabledist', [])
                if isinstance(pd, list):
                    flat.extend([float(x) for x in pd[:96]])
            
            # General info (numeric only)
            if 'general' in record:
                g = record['general']
                flat.extend([
                    float(g.get('size',              0)),
                    float(g.get('vsize',             0)),
                    float(g.get('has_debug',         0)),
                    float(g.get('exports',           0)),
                    float(g.get('imports',           0)),
                    float(g.get('has_relocations',   0)),
                    float(g.get('has_resources',     0)),
                    float(g.get('has_signature',     0)),
                    float(g.get('has_tls',           0)),
                    float(g.get('symbols',           0)),
                ])
            
            # Header optional (numeric only, skip strings)
            if 'header' in record:
                oh = record['header'].get('optional', {})
                flat.extend([
                    float(oh.get('sizeof_code',                       0)),
                    float(oh.get('sizeof_headers',                    0)),
                    float(oh.get('sizeof_heap_commit',                0)),
                    float(oh.get('major_image_version',               0)),
                    float(oh.get('minor_image_version',               0)),
                    float(oh.get('major_linker_version',              0)),
                    float(oh.get('minor_linker_version',              0)),
                    float(oh.get('major_operating_system_version',    0)),
                    float(oh.get('minor_operating_system_version',    0)),
                    float(oh.get('major_subsystem_version',           0)),
                    float(oh.get('minor_subsystem_version',           0)),
                ])
            
            # Pad or trim to exactly 634
            if len(flat) >= 634:
                flat = flat[:634]
            else:
                flat = flat + [0.0] * (634 - len(flat))
            
            features.append(flat)
            labels.append(float(label))
    
    return features, labels


# Reload all files
all_features = []
all_labels   = []

for i in range(6):
    path = f'/kaggle/working/ember2018/train_features_{i}.jsonl'
    print(f"Loading file {i}...")
    feats, labs = load_ember_numeric(path)
    all_features.extend(feats)
    all_labels.extend(labs)
    print(f"  {len(feats):,} samples")

print(f"\nTotal: {len(all_features):,} samples")

# Convert to numpy
X = np.array(all_features, dtype=np.float32)
y = np.array(all_labels,   dtype=np.float32)

print(f"Shape   : {X.shape}")
print(f"Malware : {(y==1).sum():,}")
print(f"Benign  : {(y==0).sum():,}")

# Normalize
scaler = MinMaxScaler()
X      = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

FEATURE_SIZE = X_train.shape[1]

# DataLoaders
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train),
        torch.tensor(y_train)
    ),
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test),
        torch.tensor(y_test)
    ),
    batch_size=256,
    shuffle=False
)

print(f"\nTraining samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print(f"Feature size     : {FEATURE_SIZE}")
print(f"Batches          : {len(train_loader):,}")
print("\n✅ EMBER dataset ready — pure numeric features!")

Reloading EMBER with numeric-only features...
Loading file 0...
  50,000 samples
Loading file 1...
  116,051 samples
Loading file 2...
  93,607 samples
Loading file 3...
  94,313 samples
Loading file 4...
  96,846 samples
Loading file 5...
  149,183 samples

Total: 600,000 samples
Shape   : (600000, 634)
Malware : 300,000
Benign  : 300,000

Training samples : 480,000
Testing samples  : 120,000
Feature size     : 634
Batches          : 1,875

✅ EMBER dataset ready — pure numeric features!


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Feature size: {FEATURE_SIZE}")

# --------------------------------------------------
# Detector Network (Blue Team)
# --------------------------------------------------
class Detector(nn.Module):
    def __init__(self, input_size):
        super(Detector, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)


# --------------------------------------------------
# Generator Network (Red Team)
# Deeper and stronger than Colab version
# --------------------------------------------------
class Generator(nn.Module):
    def __init__(self, input_size, noise_dim=64):
        super(Generator, self).__init__()
        self.noise_dim = noise_dim
        
        self.network = nn.Sequential(
            # Input = features + noise
            nn.Linear(input_size + noise_dim, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 2048),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(2048, 1024),
            nn.LeakyReLU(0.2),
            
            nn.Linear(1024, input_size),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # Add random noise for exploration
        noise     = torch.randn(x.size(0), self.noise_dim).to(x.device)
        x_noisy   = torch.cat([x, noise], dim=1)
        perturbed = self.network(x_noisy)
        
        # OR operation — can only ADD features
        return torch.clamp(x + perturbed, 0, 1)


# --------------------------------------------------
# Helper — Get Accuracy
# --------------------------------------------------
def get_accuracy(model, loader):
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = (model(X_batch).squeeze() >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    return correct / total


# Build networks
detector  = Detector(input_size=FEATURE_SIZE).to(device)
generator = Generator(input_size=FEATURE_SIZE).to(device)

print("\nDetector Architecture:")
print(detector)
print("\nGenerator Architecture:")
print(generator)
print("\nBoth networks built successfully!")

Device: cuda
Feature size: 634

Detector Architecture:
Detector(
  (network): Sequential(
    (0): Linear(in_features=634, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=128, out_features=64, bias=True)
    (10): ReLU()
    (11): Linear(in_features=64, out_features=1, bias=True)
    (12): Sigmoid()
  )
)

Generator Architecture:
Generator(
  (network): Sequential(
    (0): Linear(in_features=698, out_features=1024, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=1024, out_features=2048, bias=True)
    (4): LeakyReLU(negative_slope=0.2)
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=2048, out_features=1024

In [14]:
print("="*60)
print("PHASE 1 — Pretraining Detector on 480,000 EMBER samples")
print("="*60)

criterion     = nn.BCELoss()
det_optimizer = optim.Adam(detector.parameters(), lr=0.001)

for epoch in range(10):
    detector.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        
        det_optimizer.zero_grad()
        preds = detector(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        det_optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 2 == 0:
        acc = get_accuracy(detector, test_loader)
        print(f"Epoch {epoch+1}/10 → "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Accuracy: {acc*100:.2f}%")

final_acc = get_accuracy(detector, test_loader)
print(f"\nFinal Detector Accuracy: {final_acc*100:.2f}%")

if final_acc >= 0.80:
    print("✅ Detector strong enough — starting adversarial battle!")
else:
    print("⚠️ Accuracy below 80% — may need more epochs")

PHASE 1 — Pretraining Detector on 480,000 EMBER samples
Epoch 2/10 → Loss: 0.3500 | Accuracy: 85.85%
Epoch 4/10 → Loss: 0.3104 | Accuracy: 86.88%
Epoch 6/10 → Loss: 0.2925 | Accuracy: 88.00%
Epoch 8/10 → Loss: 0.2806 | Accuracy: 88.74%
Epoch 10/10 → Loss: 0.2696 | Accuracy: 88.98%

Final Detector Accuracy: 88.98%
✅ Detector strong enough — starting adversarial battle!


In [ ]:
print("="*60)
print("ADVERSARIAL BATTLE — BALANCED VERSION")
print("="*60 + "\n")

BASE_PATH        = '/kaggle/working'
evasion_history  = []
accuracy_history = []

# Fresh networks with better initialization
detector  = Detector(input_size=FEATURE_SIZE).to(device)
generator = Generator(input_size=FEATURE_SIZE).to(device)

# Pretrain detector gently to 80%
print("Pretraining Detector to ~80%...")
det_opt   = optim.Adam(detector.parameters(), lr=0.0005)
criterion = nn.BCELoss()

for epoch in range(5):
    detector.train()
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        det_opt.zero_grad()
        loss = criterion(detector(X_batch), y_batch)
        loss.backward()
        det_opt.step()

acc = get_accuracy(detector, test_loader)
print(f"Pretrain accuracy: {acc*100:.2f}%\n")

# Battle
for round_num in range(10):
    print(f"ROUND {round_num+1}/10")
    
    # Generator trains gently — just 10 epochs
    for param in detector.parameters():
        param.requires_grad = False
    
    gen_opt   = optim.Adam(generator.parameters(), lr=0.0003)
    generator.train()
    
    for epoch in range(10):
        for X_batch, y_batch in train_loader:
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            if malware_mask.sum() == 0:
                continue
            malware_samples = X_batch[malware_mask]
            
            gen_opt.zero_grad()
            perturbed = generator(malware_samples)
            score     = detector(perturbed)
            loss      = criterion(score, torch.zeros_like(score))
            loss.backward()
            gen_opt.step()
    
    # Measure evasion
    generator.eval()
    evasion_count = 0
    total_malware = 0
    
    # Only measure on 5 batches for speed
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(train_loader):
            if i >= 5:
                break
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            if malware_mask.sum() == 0:
                continue
            malware_samples  = X_batch[malware_mask]
            perturbed        = generator(malware_samples)
            scores           = detector(perturbed)
            evasion_count   += (scores < 0.5).sum().item()
            total_malware   += len(malware_samples)
    
    evasion_rate = evasion_count / max(total_malware, 1)
    evasion_history.append(evasion_rate)
    
    # Detector retrains gently — just 3 epochs
    for param in detector.parameters():
        param.requires_grad = True
    
    # Very low LR so it doesn't overcorrect
    det_opt = optim.Adam(detector.parameters(), lr=0.00005)
    
    for epoch in range(3):
        detector.train()
        for X_batch, y_batch in train_loader:
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            if malware_mask.sum() == 0:
                continue
            malware_samples = X_batch[malware_mask]
            
            det_opt.zero_grad()
            
            with torch.no_grad():
                perturbed = generator(malware_samples)
            
            benign_mask    = (y_batch.squeeze() == 0)
            benign_samples = X_batch[benign_mask]
            
            real_loss      = criterion(
                detector(malware_samples),
                torch.ones(len(malware_samples), 1).to(device)
            )
            perturbed_loss = criterion(
                detector(perturbed),
                torch.ones(len(perturbed), 1).to(device)
            )
            benign_loss = criterion(
                detector(benign_samples),
                torch.zeros(len(benign_samples), 1).to(device)
            ) if benign_samples.size(0) > 0 else torch.tensor(0.0).to(device)
            
            det_loss = (real_loss + perturbed_loss + benign_loss) / 3
            det_loss.backward()
            det_opt.step()
    
    det_accuracy = get_accuracy(detector, test_loader)
    accuracy_history.append(det_accuracy)
    
    print(f"  🔴 Evasion Rate     : {evasion_rate*100:.1f}%")
    print(f"  🔵 Detector Accuracy: {det_accuracy*100:.2f}%")
    
    # Save JSON
    results = {
        "round"             : round_num + 1,
        "evasion_rate"      : round(evasion_rate * 100, 2),
        "detector_accuracy" : round(det_accuracy * 100, 2),
        "samples_generated" : total_malware,
        "threshold_breached": evasion_rate > 0.2,
        "timestamp"         : str(datetime.now()),
        "alerts"            : ["Retrain triggered"] if evasion_rate > 0.2 else [],
        "training_status"   : "running",
        "evasion_history"   : [round(e*100,2) for e in evasion_history],
        "accuracy_history"  : [round(a*100,2) for a in accuracy_history]
    }
    with open(f'{BASE_PATH}/results.json', 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"  💾 Saved!\n")

print("="*60)
print("BATTLE COMPLETE")
print("="*60)
print(f"Evasion History  : {[f'{e*100:.1f}%' for e in evasion_history]}")
print(f"Accuracy History : {[f'{a*100:.1f}%' for a in accuracy_history]}")
print(f"Peak Evasion     : {max(evasion_history)*100:.1f}%")

# Final save
results['training_status'] = 'complete'
results['evasion_history']  = [round(e*100,2) for e in evasion_history]
results['accuracy_history'] = [round(a*100,2) for a in accuracy_history]
with open(f'{BASE_PATH}/results.json', 'w') as f:
    json.dump(results, f, indent=4)
print("✅ Final results.json saved!")

In [1]:
# Lightweight accuracy check
def get_accuracy_fast(model, loader, max_batches=4):
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(loader):
            if i >= max_batches:
                break
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = (model(X_batch).squeeze() >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    return correct / total

print("="*60)
print("ADVERSARIAL BATTLE — LIGHTWEIGHT VERSION")
print("="*60 + "\n")

BASE_PATH = '/kaggle/working'

# Use 20k samples for battle
BATTLE_SIZE = 20000
idx         = torch.randperm(len(X_train))[:BATTLE_SIZE]
X_battle    = torch.tensor(X_train[idx.numpy()])
y_battle    = torch.tensor(y_train[idx.numpy()])

battle_loader = DataLoader(
    TensorDataset(X_battle, y_battle),
    batch_size=512,
    shuffle=True
)

print(f"Battle size : {BATTLE_SIZE:,} samples")
print(f"Batches     : {len(battle_loader)}\n")

# Fresh networks
detector  = Detector(input_size=FEATURE_SIZE).to(device)
generator = Generator(input_size=FEATURE_SIZE).to(device)
criterion = nn.BCELoss()

# Quick pretrain — 3 epochs only
print("Quick pretraining Detector...")
det_opt = optim.Adam(detector.parameters(), lr=0.001)

for epoch in range(3):
    detector.train()
    for X_batch, y_batch in battle_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        det_opt.zero_grad()
        loss = criterion(detector(X_batch), y_batch)
        loss.backward()
        det_opt.step()

acc = get_accuracy_fast(detector, battle_loader)
print(f"Pretrain accuracy: {acc*100:.2f}%\n")
print("Starting battle...\n")

evasion_history  = []
accuracy_history = []

for round_num in range(10):
    
    # Generator attacks — 3 epochs
    for param in detector.parameters():
        param.requires_grad = False
    
    gen_opt = optim.Adam(generator.parameters(), lr=0.0005)
    generator.train()
    
    for epoch in range(3):
        for X_batch, y_batch in battle_loader:
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            if malware_mask.sum() == 0:
                continue
            malware_samples = X_batch[malware_mask]
            gen_opt.zero_grad()
            perturbed       = generator(malware_samples)
            score           = detector(perturbed)
            loss            = criterion(score, torch.zeros_like(score))
            loss.backward()
            gen_opt.step()
    
    # Measure evasion on 2 batches
    generator.eval()
    evasion_count = 0
    total_malware = 0
    
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(battle_loader):
            if i >= 2:
                break
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            if malware_mask.sum() == 0:
                continue
            malware_samples  = X_batch[malware_mask]
            perturbed        = generator(malware_samples)
            scores           = detector(perturbed)
            evasion_count   += (scores < 0.5).sum().item()
            total_malware   += len(malware_samples)
    
    evasion_rate = evasion_count / max(total_malware, 1)
    evasion_history.append(evasion_rate)
    
    # Detector retrains — 2 epochs
    for param in detector.parameters():
        param.requires_grad = True
    
    det_opt = optim.Adam(detector.parameters(), lr=0.00005)
    
    for epoch in range(2):
        detector.train()
        for X_batch, y_batch in battle_loader:
            X_batch      = X_batch.to(device)
            y_batch      = y_batch.to(device).unsqueeze(1)
            malware_mask = (y_batch.squeeze() == 1)
            benign_mask  = (y_batch.squeeze() == 0)
            if malware_mask.sum() == 0:
                continue
            malware_samples = X_batch[malware_mask]
            benign_samples  = X_batch[benign_mask]
            det_opt.zero_grad()
            with torch.no_grad():
                perturbed = generator(malware_samples)
            real_loss = criterion(
                detector(malware_samples),
                torch.ones(len(malware_samples), 1).to(device)
            )
            perturbed_loss = criterion(
                detector(perturbed),
                torch.ones(len(perturbed), 1).to(device)
            )
            benign_loss = criterion(
                detector(benign_samples),
                torch.zeros(len(benign_samples), 1).to(device)
            ) if benign_samples.size(0) > 0 else torch.tensor(0.0).to(device)
            det_loss = (real_loss + perturbed_loss + benign_loss) / 3
            det_loss.backward()
            det_opt.step()
    
    det_accuracy = get_accuracy_fast(detector, battle_loader)
    accuracy_history.append(det_accuracy)
    
    print(f"Round {round_num+1:2d}/10 | "
          f"🔴 Evasion: {evasion_rate*100:5.1f}% | "
          f"🔵 Detector: {det_accuracy*100:.2f}%")
    
    results = {
        "round"             : round_num + 1,
        "evasion_rate"      : round(evasion_rate * 100, 2),
        "detector_accuracy" : round(det_accuracy * 100, 2),
        "samples_generated" : total_malware,
        "threshold_breached": evasion_rate > 0.2,
        "timestamp"         : str(datetime.now()),
        "alerts"            : ["Retrain triggered"] if evasion_rate > 0.2 else [],
        "training_status"   : "running",
        "evasion_history"   : [round(e*100,2) for e in evasion_history],
        "accuracy_history"  : [round(a*100,2) for a in accuracy_history]
    }
    with open(f'{BASE_PATH}/results.json', 'w') as f:
        json.dump(results, f, indent=4)

print("\n" + "="*60)
print("BATTLE COMPLETE")
print("="*60)
print(f"Evasion  : {[f'{e*100:.1f}%' for e in evasion_history]}")
print(f"Accuracy : {[f'{a*100:.1f}%' for a in accuracy_history]}")
print(f"Peak Evasion : {max(evasion_history)*100:.1f}%")

results['training_status'] = 'complete'
with open(f'{BASE_PATH}/results.json', 'w') as f:
    json.dump(results, f, indent=4)
print("✅ Done!")

ADVERSARIAL BATTLE — LIGHTWEIGHT VERSION



NameError: name 'torch' is not defined

In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("All imports done!")

Device: cuda
All imports done!


In [3]:
def load_ember_numeric(filepath):
    features = []
    labels   = []
    with open(filepath, 'r') as f:
        for line in f:
            record = json.loads(line)
            label  = record.get('label', -1)
            if label == -1:
                continue
            flat = []
            if 'histogram' in record:
                flat.extend([float(x) for x in record['histogram'][:256]])
            if 'byteentropy' in record:
                flat.extend([float(x) for x in record['byteentropy'][:256]])
            if 'strings' in record:
                s = record['strings']
                flat.append(float(s.get('numstrings', 0)))
                flat.append(float(s.get('avlength',   0)))
                flat.append(float(s.get('entropy',    0)))
                flat.append(float(s.get('paths',      0)))
                flat.append(float(s.get('urls',       0)))
                flat.append(float(s.get('registry',   0)))
                flat.append(float(s.get('MZ',         0)))
                pd = s.get('printabledist', [])
                if isinstance(pd, list):
                    flat.extend([float(x) for x in pd[:96]])
            if 'general' in record:
                g = record['general']
                flat.extend([
                    float(g.get('size',            0)),
                    float(g.get('vsize',           0)),
                    float(g.get('has_debug',       0)),
                    float(g.get('exports',         0)),
                    float(g.get('imports',         0)),
                    float(g.get('has_relocations', 0)),
                    float(g.get('has_resources',   0)),
                    float(g.get('has_signature',   0)),
                    float(g.get('has_tls',         0)),
                    float(g.get('symbols',         0)),
                ])
            if 'header' in record:
                oh = record['header'].get('optional', {})
                flat.extend([
                    float(oh.get('sizeof_code',                    0)),
                    float(oh.get('sizeof_headers',                 0)),
                    float(oh.get('sizeof_heap_commit',             0)),
                    float(oh.get('major_image_version',            0)),
                    float(oh.get('minor_image_version',            0)),
                    float(oh.get('major_linker_version',           0)),
                    float(oh.get('minor_linker_version',           0)),
                    float(oh.get('major_operating_system_version', 0)),
                    float(oh.get('minor_operating_system_version', 0)),
                    float(oh.get('major_subsystem_version',        0)),
                    float(oh.get('minor_subsystem_version',        0)),
                ])
            if len(flat) >= 634:
                flat = flat[:634]
            else:
                flat = flat + [0.0] * (634 - len(flat))
            features.append(flat)
            labels.append(float(label))
    return features, labels

print("Loading EMBER dataset...")
all_features = []
all_labels   = []

for i in range(6):
    path = f'/kaggle/working/ember2018/train_features_{i}.jsonl'
    print(f"Loading file {i}...")
    feats, labs = load_ember_numeric(path)
    all_features.extend(feats)
    all_labels.extend(labs)
    print(f"  {len(feats):,} samples")

X = np.array(all_features, dtype=np.float32)
y = np.array(all_labels,   dtype=np.float32)

scaler      = MinMaxScaler()
X           = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
FEATURE_SIZE = X_train.shape[1]

print(f"\nTotal    : {X.shape}")
print(f"Train    : {len(X_train):,}")
print(f"Test     : {len(X_test):,}")
print(f"Features : {FEATURE_SIZE}")
print("Dataset ready!")

Loading EMBER dataset...
Loading file 0...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/ember2018/train_features_0.jsonl'

In [4]:
import os

data_path = '/kaggle/working/ember2018/train_features_0.jsonl'
archive_url = 'https://ember.elastic.co/ember_dataset_2018_2.tar.bz2'
archive_name = 'ember_dataset_2018_2.tar.bz2'

if os.path.exists(data_path):
    print("✅ EMBER files found! You are good to run Cell 2.")
else:
    print("⚠️ Kaggle wiped your working directory. Re-downloading EMBER...")
    
    # Download if the tar file doesn't exist
    if not os.path.exists(archive_name):
        !wget {archive_url}
        
    print("📦 Extracting EMBER dataset... (This will take a minute or two)")
    # Extract directly to /kaggle/working/ (the tar creates the ember2018 folder automatically)
    !tar -xjvf {archive_name} -C /kaggle/working/
    
    print("\n✅ Extraction complete! Now you can run Cell 2.")

⚠️ Kaggle wiped your working directory. Re-downloading EMBER...
--2026-04-30 11:50:07--  https://ember.elastic.co/ember_dataset_2018_2.tar.bz2
Resolving ember.elastic.co (ember.elastic.co)... 34.107.161.234, 2600:1901:0:1f6d::
Connecting to ember.elastic.co (ember.elastic.co)|34.107.161.234|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1696539273 (1.6G) [application/x-bzip2]
Saving to: ‘ember_dataset_2018_2.tar.bz2’

ember_dataset_2018_ 100%[===================>]   1.58G  39.0MB/s    in 37s     

2026-04-30 11:50:45 (43.2 MB/s) - ‘ember_dataset_2018_2.tar.bz2’ saved [1696539273/1696539273]

📦 Extracting EMBER dataset... (This will take a minute or two)
ember2018/
ember2018/train_features_1.jsonl
ember2018/train_features_0.jsonl
ember2018/train_features_3.jsonl
ember2018/test_features.jsonl
ember2018/ember_model_2018.txt
ember2018/train_features_5.jsonl
ember2018/train_features_4.jsonl
ember2018/train_features_2.jsonl

✅ Extraction complete! Now you can run C

In [5]:
def load_ember_numeric(filepath):
    features = []
    labels   = []
    with open(filepath, 'r') as f:
        for line in f:
            record = json.loads(line)
            label  = record.get('label', -1)
            if label == -1:
                continue
            flat = []
            if 'histogram' in record:
                flat.extend([float(x) for x in record['histogram'][:256]])
            if 'byteentropy' in record:
                flat.extend([float(x) for x in record['byteentropy'][:256]])
            if 'strings' in record:
                s = record['strings']
                flat.append(float(s.get('numstrings', 0)))
                flat.append(float(s.get('avlength',   0)))
                flat.append(float(s.get('entropy',    0)))
                flat.append(float(s.get('paths',      0)))
                flat.append(float(s.get('urls',       0)))
                flat.append(float(s.get('registry',   0)))
                flat.append(float(s.get('MZ',         0)))
                pd = s.get('printabledist', [])
                if isinstance(pd, list):
                    flat.extend([float(x) for x in pd[:96]])
            if 'general' in record:
                g = record['general']
                flat.extend([
                    float(g.get('size',            0)),
                    float(g.get('vsize',           0)),
                    float(g.get('has_debug',       0)),
                    float(g.get('exports',         0)),
                    float(g.get('imports',         0)),
                    float(g.get('has_relocations', 0)),
                    float(g.get('has_resources',   0)),
                    float(g.get('has_signature',   0)),
                    float(g.get('has_tls',         0)),
                    float(g.get('symbols',         0)),
                ])
            if 'header' in record:
                oh = record['header'].get('optional', {})
                flat.extend([
                    float(oh.get('sizeof_code',                    0)),
                    float(oh.get('sizeof_headers',                 0)),
                    float(oh.get('sizeof_heap_commit',             0)),
                    float(oh.get('major_image_version',            0)),
                    float(oh.get('minor_image_version',            0)),
                    float(oh.get('major_linker_version',           0)),
                    float(oh.get('minor_linker_version',           0)),
                    float(oh.get('major_operating_system_version', 0)),
                    float(oh.get('minor_operating_system_version', 0)),
                    float(oh.get('major_subsystem_version',        0)),
                    float(oh.get('minor_subsystem_version',        0)),
                ])
            if len(flat) >= 634:
                flat = flat[:634]
            else:
                flat = flat + [0.0] * (634 - len(flat))
            features.append(flat)
            labels.append(float(label))
    return features, labels

print("Loading EMBER dataset...")
all_features = []
all_labels   = []

for i in range(6):
    path = f'/kaggle/working/ember2018/train_features_{i}.jsonl'
    print(f"Loading file {i}...")
    feats, labs = load_ember_numeric(path)
    all_features.extend(feats)
    all_labels.extend(labs)
    print(f"  {len(feats):,} samples")

X = np.array(all_features, dtype=np.float32)
y = np.array(all_labels,   dtype=np.float32)

scaler      = MinMaxScaler()
X           = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
FEATURE_SIZE = X_train.shape[1]

print(f"\nTotal    : {X.shape}")
print(f"Train    : {len(X_train):,}")
print(f"Test     : {len(X_test):,}")
print(f"Features : {FEATURE_SIZE}")
print("Dataset ready!")

Loading EMBER dataset...
Loading file 0...
  50,000 samples
Loading file 1...
  116,051 samples
Loading file 2...
  93,607 samples
Loading file 3...
  94,313 samples
Loading file 4...
  96,846 samples
Loading file 5...
  149,183 samples

Total    : (600000, 634)
Train    : 480,000
Test     : 120,000
Features : 634
Dataset ready!


In [2]:
class Detector(nn.Module):
    def __init__(self, input_size):
        super(Detector, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),         nn.ReLU(),
            nn.Linear(64, 1),           nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

class Generator(nn.Module):
    def __init__(self, input_size, noise_dim=64):
        super(Generator, self).__init__()
        self.noise_dim = noise_dim
        self.network = nn.Sequential(
            nn.Linear(input_size + noise_dim, 1024), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(1024, 2048),                   nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(2048, 1024),                   nn.LeakyReLU(0.2),
            nn.Linear(1024, input_size),             nn.Sigmoid()
        )
    def forward(self, x):
        noise     = torch.randn(x.size(0), self.noise_dim).to(x.device)
        x_noisy   = torch.cat([x, noise], dim=1)
        perturbed = self.network(x_noisy)
        return torch.clamp(x + perturbed, 0, 1)

def get_accuracy_fast(model, loader, max_batches=4):
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(loader):
            if i >= max_batches:
                break
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = (model(X_batch).squeeze() >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    return correct / total

detector  = Detector(input_size=FEATURE_SIZE).to(device)
generator = Generator(input_size=FEATURE_SIZE).to(device)
print("Networks ready!")

NameError: name 'nn' is not defined

In [1]:
# ==========================================
# STEP 1: INITIAL SETUP & PRETRAIN
# ==========================================
BATTLE_SIZE = 20000
BATCH_SIZE = 512

def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad = requires_grad

set_requires_grad(detector, True)
set_requires_grad(generator, True)

print("Sampling Initial 20k subset for Pretraining...")
indices = np.random.choice(len(X_train), BATTLE_SIZE, replace=False)
X_battle, y_battle = X_train[indices], y_train[indices]
battle_loader = DataLoader(TensorDataset(torch.tensor(X_battle), torch.tensor(y_battle)), batch_size=BATCH_SIZE, shuffle=True)

optimizer_pretrain = optim.Adam(detector.parameters(), lr=0.001) 
criterion = nn.BCELoss()

print("Pretraining Detector for 3 epochs...")
detector.train()
for epoch in range(3):
    for batch_x, batch_y in battle_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer_pretrain.zero_grad()
        loss_D = criterion(detector(batch_x).squeeze(), batch_y)
        loss_D.backward()
        optimizer_pretrain.step()

detector.eval()
initial_acc = get_accuracy_fast(detector, battle_loader, max_batches=4)
display_initial = initial_acc * 100 if initial_acc <= 1.0 else initial_acc
print(f"Detector Pretrain Complete. Initial Accuracy: {display_initial:.2f}%\n")

# ==========================================
# STEP 3 & 4: ADVANCED ADVERSARIAL BATTLE
# ==========================================
optimizer_D = optim.Adam(detector.parameters(), lr=0.00005) 
optimizer_G = optim.Adam(generator.parameters(), lr=0.0005)

evasion_history = []
accuracy_history = []

print("Starting Advanced Adversarial Arms Race (10 Rounds)...")
for round_idx in range(1, 11):
    
    # UPGRADE 1: ROLLING BATTLEGROUND
    # Resample a brand new 20k files every single round!
    indices = np.random.choice(len(X_train), BATTLE_SIZE, replace=False)
    X_battle, y_battle = X_train[indices], y_train[indices]
    battle_loader = DataLoader(TensorDataset(torch.tensor(X_battle), torch.tensor(y_battle)), batch_size=BATCH_SIZE, shuffle=True)
    
    # ----------------------------------------------------
    # A) Generator trains (Detector frozen)
    # ----------------------------------------------------
    set_requires_grad(detector, False)
    set_requires_grad(generator, True)
    generator.train()
    detector.eval()
    
    for _ in range(3): 
        for batch_x, batch_y in battle_loader:
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            if len(malware_idx) == 0: continue
                
            real_malware = batch_x[malware_idx].to(device)
            optimizer_G.zero_grad()
            
            adversarial_malware = generator(real_malware) 
            predictions = detector(adversarial_malware).squeeze()
            target_benign = torch.empty_like(predictions).uniform_(0.1, 0.3)
            bce_loss = criterion(predictions, target_benign)
            
            # UPGRADE 2: L1 "SNIPER" REGULARIZATION
            # L2 keeps changes small. L1 forces changes to be SPARSE (fewer features modified, but modified heavily)
            l2_penalty = torch.mean((adversarial_malware - real_malware)**2)
            l1_penalty = torch.mean(torch.abs(adversarial_malware - real_malware))
            
            # Combine them: 0.05 L2 for general stability, 0.02 L1 to force sparse real-world hacking behavior
            loss_G = bce_loss + (0.05 * l2_penalty) + (0.02 * l1_penalty)
            
            loss_G.backward()
            optimizer_G.step()

    # ----------------------------------------------------
    # B) Measure Evasion on 2 Batches
    # ----------------------------------------------------
    generator.eval()
    evaded_count, total_malware = 0, 0
    with torch.no_grad():
        for i, (batch_x, batch_y) in enumerate(battle_loader):
            if i >= 2: break 
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            if len(malware_idx) == 0: continue
                
            real_malware = batch_x[malware_idx].to(device)
            adversarial_malware = generator(real_malware)
            preds = detector(adversarial_malware).squeeze()
            
            evaded_count += (preds < 0.5).sum().item()
            total_malware += len(real_malware)
            
    evasion_rate = (evaded_count / total_malware * 100) if total_malware > 0 else 0
    evasion_history.append(evasion_rate)

    # ----------------------------------------------------
    # C) Detector retrains 2 epochs (Generator frozen)
    # ----------------------------------------------------
    set_requires_grad(detector, True)
    set_requires_grad(generator, False)
    detector.train()
    
    for _ in range(2): 
        for batch_x, batch_y in battle_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            
            if len(malware_idx) > 1:
                split_idx = len(malware_idx) // 2
                adv_idx = malware_idx[:split_idx]
                real_malware = batch_x[adv_idx]
                
                with torch.no_grad():
                    adversarial_malware = generator(real_malware)
                
                batch_x[adv_idx] = adversarial_malware
                
            soft_targets = batch_y.float()
            soft_targets[soft_targets == 1.0] = 0.9 
                
            optimizer_D.zero_grad()
            outputs = detector(batch_x).squeeze()
            loss_D = criterion(outputs, soft_targets)
            loss_D.backward()
            optimizer_D.step()

    # ----------------------------------------------------
    # D & F) Accuracy Check and Logging
    # ----------------------------------------------------
    detector.eval()
    raw_acc = get_accuracy_fast(detector, battle_loader, max_batches=4)
    display_acc = raw_acc * 100 if raw_acc <= 1.0 else raw_acc
    accuracy_history.append(display_acc)

    print(f"Round {round_idx:2d}/10 | Evasion: {evasion_rate:5.1f}% | Detector: {display_acc:5.2f}%")

    # ----------------------------------------------------
    # E) Save results.json
    # ----------------------------------------------------
    threshold_breached = bool(evasion_rate > 20.0)
    alerts = ["Retrain triggered"] if threshold_breached else []
    
    results = {
        "round": round_idx,
        "evasion_rate": round(evasion_rate, 2),
        "detector_accuracy": round(display_acc, 2),
        "samples_generated": BATTLE_SIZE,
        "threshold_breached": threshold_breached,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "alerts": alerts,
        "training_status": "running" if round_idx < 10 else "completed",
        "evasion_history": [round(x, 2) for x in evasion_history],
        "accuracy_history": [round(x, 2) for x in accuracy_history]
    }
    with open("/kaggle/working/results.json", "w") as f:
        json.dump(results, f, indent=4)
        
print("\nBattle Complete! File saved to /kaggle/working/results.json ready for CI/CD Pipeline.")

NameError: name 'detector' is not defined

In [4]:
!rm -rf /kaggle/working/ember.tar.bz2 /kaggle/working/ember2018
print("Cleaned up corrupted files!")

Cleaned up corrupted files!


In [5]:
import os

archive_url = 'https://ember.elastic.co/ember_dataset_2018_2.tar.bz2'
archive_name = 'ember_dataset_2018_2.tar.bz2'

print("⚠️ Re-downloading EMBER safely...")
if not os.path.exists(archive_name):
    !wget {archive_url}
    
print("📦 Extracting EMBER dataset... (Wait for this to completely finish!)")
!tar -xjvf {archive_name} -C /kaggle/working/

print("\n✅ Extraction complete! All files are ready.")

⚠️ Re-downloading EMBER safely...
--2026-04-30 16:01:05--  https://ember.elastic.co/ember_dataset_2018_2.tar.bz2
Resolving ember.elastic.co (ember.elastic.co)... 34.107.161.234, 2600:1901:0:1f6d::
Connecting to ember.elastic.co (ember.elastic.co)|34.107.161.234|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1696539273 (1.6G) [application/x-bzip2]
Saving to: ‘ember_dataset_2018_2.tar.bz2’

ember_dataset_2018_ 100%[===================>]   1.58G  8.22MB/s    in 3m 33s  

2026-04-30 16:04:39 (7.60 MB/s) - ‘ember_dataset_2018_2.tar.bz2’ saved [1696539273/1696539273]

📦 Extracting EMBER dataset... (Wait for this to completely finish!)
ember2018/
ember2018/train_features_1.jsonl
ember2018/train_features_0.jsonl
ember2018/train_features_3.jsonl
ember2018/test_features.jsonl
ember2018/ember_model_2018.txt
ember2018/train_features_5.jsonl
ember2018/train_features_4.jsonl
ember2018/train_features_2.jsonl

✅ Extraction complete! All files are ready.


In [6]:
# ==========================================
# CELL 1: SETUP & FAILSAFE DOWNLOADER
# ==========================================
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import os

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Libraries loaded! Using device: {device}")

# Failsafe dataset downloader
archive_url = 'https://ember.elastic.co/ember_dataset_2018_2.tar.bz2'
archive_name = 'ember_dataset_2018_2.tar.bz2'
data_dir = '/kaggle/working/ember2018'

if not os.path.exists(f"{data_dir}/train_features_0.jsonl"):
    print("⚠️ Dataset not found in working directory. Downloading EMBER safely...")
    if not os.path.exists(archive_name):
        os.system(f'wget {archive_url}')
        
    print("📦 Extracting EMBER dataset... (This takes about 2 minutes. Wait for the green checkmark!)")
    os.system(f'tar -xjf {archive_name} -C /kaggle/working/')
    print("✅ Extraction complete! All files are ready.")
else:
    print("✅ EMBER files found on disk!")

✅ Libraries loaded! Using device: cuda
✅ EMBER files found on disk!


In [7]:
# ==========================================
# CELL 2: EMBER DATASET LOADER (Wait 5-8 mins)
# ==========================================
FEATURE_SIZE = 634

def load_ember_numeric(filepath):
    """Extracts only the 634 numeric features you specified"""
    features, labels = [], []
    with open(filepath, 'r') as f:
        for line in f:
            record = json.loads(line)
            if record['label'] == -1: continue # Skip unlabelled
            
            # Extract the exactly 634 features (Histogram: 256, Entropy: 256, Strings: 103, General: 10, Header: 11)
            # Assuming you flatten the dicts/lists into a single array
            hist = record.get('histogram', [0]*256)
            ent = record.get('byteentropy', [0]*256)
            
            str_info = record.get('strings', {})
            str_stats = [str_info.get(k, 0) for k in ['numstrings', 'avlength', 'printables', 'entropy', 'paths', 'urls', 'registry', 'MZ']]
            str_printables = str_info.get('printabledist', [0]*96) # 96 printables
            str_features = str_stats[:7] + str_printables # 7 + 96 = 103 features
            
            gen = record.get('general', {})
            gen_features = [gen.get(k, 0) for k in ['size', 'vsize', 'has_debug', 'exports', 'imports', 'has_reloc', 'has_resources', 'has_signature', 'has_tls', 'symbols']]
            
            hdr = record.get('header', {}).get('optional', {})
            hdr_features = [hdr.get(k, 0) for k in ['major_image_version', 'minor_image_version', 'major_linker_version', 'minor_linker_version', 'major_operating_system_version', 'minor_operating_system_version', 'major_subsystem_version', 'minor_subsystem_version', 'sizeof_code', 'sizeof_headers', 'sizeof_heap_commit']]
            
            # Combine all 634
            row = hist + ent + str_features + gen_features + hdr_features
            features.append(row[:FEATURE_SIZE]) # Ensure exactly 634
            labels.append(record['label'])
            
    return features, labels

print("Loading EMBER dataset into RAM... Grab a coffee ☕")
all_features, all_labels = [], []

# Load training files
for i in range(6):
    path = f'/kaggle/working/ember2018/train_features_{i}.jsonl'
    print(f"Loading file {i}/5...")
    if os.path.exists(path):
        feats, labs = load_ember_numeric(path)
        all_features.extend(feats)
        all_labels.extend(labs)

# Scale features to [0, 1]
print("Normalizing features...")
scaler = MinMaxScaler()
X_train = scaler.fit_transform(np.array(all_features, dtype=np.float32))
y_train = np.array(all_labels, dtype=np.float32)

print(f"✅ Dataset Loaded! Total Training Samples: {len(X_train)} | Features: {X_train.shape[1]}")

Loading EMBER dataset into RAM... Grab a coffee ☕
Loading file 0/5...
Loading file 1/5...
Loading file 2/5...
Loading file 3/5...
Loading file 4/5...
Loading file 5/5...
Normalizing features...
✅ Dataset Loaded! Total Training Samples: 600000 | Features: 634


In [8]:
# ==========================================
# CELL 3: NETWORKS & UTILITIES
# ==========================================
class Detector(nn.Module):
    def __init__(self, input_size):
        super(Detector, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.network(x)

class Generator(nn.Module):
    def __init__(self, feature_size, noise_dim=64):
        super(Generator, self).__init__()
        self.noise_dim = noise_dim
        self.network = nn.Sequential(
            nn.Linear(feature_size + noise_dim, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, 2048), nn.LeakyReLU(0.2),
            nn.Linear(2048, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, feature_size)
        )
        
    def forward(self, x):
        # Generator injects noise and clamps internally
        noise = torch.randn(x.size(0), self.noise_dim).to(x.device)
        x_noisy = torch.cat([x, noise], dim=1)
        perturbed = self.network(x_noisy)
        return torch.clamp(x + perturbed, 0.0, 1.0) # Logical OR simulation

def get_accuracy_fast(model, loader, max_batches=4):
    """Speedy accuracy check to prevent Kaggle freezing"""
    correct = 0
    total = 0
    with torch.no_grad():
        for i, (batch_x, batch_y) in enumerate(loader):
            if i >= max_batches: break
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x).squeeze()
            predictions = (outputs >= 0.5).float()
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
    return correct / total if total > 0 else 0

# Initialize networks
detector = Detector(FEATURE_SIZE).to(device)
generator = Generator(FEATURE_SIZE).to(device)
print("✅ Networks initialized and sent to GPU!")

✅ Networks initialized and sent to GPU!


In [11]:
# ==========================================
# CELL 4: ADVANCED ADVERSARIAL BATTLE LOOP
# ==========================================
BATTLE_SIZE = 20000
BATCH_SIZE = 512

def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad = requires_grad

# Reset state just in case
set_requires_grad(detector, True)
set_requires_grad(generator, True)

print("Sampling Initial 20k subset for Pretraining...")
indices = np.random.choice(len(X_train), BATTLE_SIZE, replace=False)
X_battle, y_battle = X_train[indices], y_train[indices]
battle_loader = DataLoader(TensorDataset(torch.tensor(X_battle), torch.tensor(y_battle)), batch_size=BATCH_SIZE, shuffle=True)

optimizer_pretrain = optim.Adam(detector.parameters(), lr=0.001) 
criterion = nn.BCELoss()

print("Pretraining Detector for 3 epochs...")
detector.train()
for epoch in range(3):
    for batch_x, batch_y in battle_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer_pretrain.zero_grad()
        loss_D = criterion(detector(batch_x).squeeze(), batch_y)
        loss_D.backward()
        optimizer_pretrain.step()

detector.eval()
initial_acc = get_accuracy_fast(detector, battle_loader, max_batches=4)
display_initial = initial_acc * 100 if initial_acc <= 1.0 else initial_acc
print(f"✅ Detector Pretrain Complete. Initial Accuracy: {display_initial:.2f}%\n")

# ==========================================
# THE ARMS RACE
# ==========================================
optimizer_D = optim.Adam(detector.parameters(), lr=0.00005) 
optimizer_G = optim.Adam(generator.parameters(), lr=0.0005)

evasion_history = []
accuracy_history = []

print("Starting Advanced Adversarial Arms Race (10 Rounds)...")
for round_idx in range(1, 11):
    
    # UPGRADE 1: ROLLING BATTLEGROUND (New files every round)
    indices = np.random.choice(len(X_train), BATTLE_SIZE, replace=False)
    X_battle, y_battle = X_train[indices], y_train[indices]
    battle_loader = DataLoader(TensorDataset(torch.tensor(X_battle), torch.tensor(y_battle)), batch_size=BATCH_SIZE, shuffle=True)
    
 # --- RED TEAM (Generator) ---
    set_requires_grad(detector, False)
    set_requires_grad(generator, True)
    generator.train()
    detector.eval()
    
    # Dropped to 1 epoch. The Generator is too smart and needs less time to think.
    for _ in range(1): 
        for batch_x, batch_y in battle_loader:
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            if len(malware_idx) == 0: continue
                
            real_malware = batch_x[malware_idx].to(device)
            optimizer_G.zero_grad()
            
            adversarial_malware = generator(real_malware) 
            predictions = detector(adversarial_malware).squeeze()
            
            # Keep soft targets
            target_benign = torch.empty_like(predictions).uniform_(0.1, 0.3)
            bce_loss = criterion(predictions, target_benign)
            
            # Massive increase to penalty weights (0.5). 
            # If the Red Team wants to mutate a feature, they have to really earn it.
            l2_diff = torch.sum((adversarial_malware - real_malware)**2, dim=1).mean()
            l1_diff = torch.sum(torch.abs(adversarial_malware - real_malware), dim=1).mean()
            
            loss_G = bce_loss + (0.5 * l2_diff) + (0.5 * l1_diff)
            
            loss_G.backward()
            optimizer_G.step()
    # --- MEASURE EVASION ---
    generator.eval()
    evaded_count, total_malware = 0, 0
    with torch.no_grad():
        for i, (batch_x, batch_y) in enumerate(battle_loader):
            if i >= 2: break 
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            if len(malware_idx) == 0: continue
                
            real_malware = batch_x[malware_idx].to(device)
            adversarial_malware = generator(real_malware)
            preds = detector(adversarial_malware).squeeze()
            
            evaded_count += (preds < 0.5).sum().item()
            total_malware += len(real_malware)
            
    evasion_rate = (evaded_count / total_malware * 100) if total_malware > 0 else 0
    evasion_history.append(evasion_rate)

    # --- BLUE TEAM (Detector) ---
    set_requires_grad(detector, True)
    set_requires_grad(generator, False)
    detector.train()
    
    for _ in range(2): 
        for batch_x, batch_y in battle_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            malware_idx = (batch_y == 1).nonzero(as_tuple=True)[0]
            
            if len(malware_idx) > 1:
                split_idx = len(malware_idx) // 2
                adv_idx = malware_idx[:split_idx]
                real_malware = batch_x[adv_idx]
                
                with torch.no_grad():
                    adversarial_malware = generator(real_malware)
                
                batch_x[adv_idx] = adversarial_malware
                
            soft_targets = batch_y.float()
            soft_targets[soft_targets == 1.0] = 0.9 
                
            optimizer_D.zero_grad()
            outputs = detector(batch_x).squeeze()
            loss_D = criterion(outputs, soft_targets)
            loss_D.backward()
            optimizer_D.step()

    # --- SAVE AND PRINT RESULTS ---
    detector.eval()
    raw_acc = get_accuracy_fast(detector, battle_loader, max_batches=4)
    display_acc = raw_acc * 100 if raw_acc <= 1.0 else raw_acc
    accuracy_history.append(display_acc)

    print(f"Round {round_idx:2d}/10 | Evasion: {evasion_rate:5.1f}% | Detector: {display_acc:5.2f}%")

    threshold_breached = bool(evasion_rate > 20.0)
    results = {
        "round": round_idx,
        "evasion_rate": round(evasion_rate, 2),
        "detector_accuracy": round(display_acc, 2),
        "samples_generated": BATTLE_SIZE,
        "threshold_breached": threshold_breached,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "alerts": ["Retrain triggered"] if threshold_breached else [],
        "training_status": "running" if round_idx < 10 else "completed",
        "evasion_history": [round(x, 2) for x in evasion_history],
        "accuracy_history": [round(x, 2) for x in accuracy_history]
    }
    with open("/kaggle/working/results.json", "w") as f:
        json.dump(results, f, indent=4)
        
print("\n🏆 Battle Complete! File saved to /kaggle/working/results.json ready for CI/CD Pipeline.")

Sampling Initial 20k subset for Pretraining...
Pretraining Detector for 3 epochs...
✅ Detector Pretrain Complete. Initial Accuracy: 79.10%

Starting Advanced Adversarial Arms Race (10 Rounds)...
Round  1/10 | Evasion:  99.6% | Detector: 77.39%
Round  2/10 | Evasion:  99.0% | Detector: 77.15%
Round  3/10 | Evasion:  99.4% | Detector: 76.46%
Round  4/10 | Evasion:  62.1% | Detector: 75.59%
Round  5/10 | Evasion:  76.9% | Detector: 75.63%
Round  6/10 | Evasion:  28.5% | Detector: 78.52%
Round  7/10 | Evasion:  74.5% | Detector: 74.07%
Round  8/10 | Evasion:  27.5% | Detector: 77.98%
Round  9/10 | Evasion:  78.6% | Detector: 77.25%
Round 10/10 | Evasion:  31.2% | Detector: 79.69%

🏆 Battle Complete! File saved to /kaggle/working/results.json ready for CI/CD Pipeline.


In [12]:
from IPython.display import FileLink
FileLink(r'results.json')

/kaggle/working/results.json

In [13]:
import joblib
import torch
from IPython.display import FileLink, display

# 1. Save the Scikit-Learn Scaler as a .pkl
joblib.dump(scaler, 'scaler.pkl')
print("✅ scaler.pkl saved!")

# 2. Save the PyTorch Detector weights as a .pth
torch.save(detector.state_dict(), 'malware_detector.pth')
print("✅ malware_detector.pth saved!")

# 3. Generate direct download links
print("\n👇 Click these links to download your files:")
display(FileLink('scaler.pkl'))
display(FileLink('malware_detector.pth'))

✅ scaler.pkl saved!
✅ malware_detector.pth saved!

👇 Click these links to download your files:


/kaggle/working/scaler.pkl

/kaggle/working/malware_detector.pth